# Task 4: Model Creation

Three models, one per authentication/recommendation stage in the system diagram:

1. **Facial Recognition Model** - classifies which member (M1-M4) a face belongs to, from `image_features.csv`.
2. **Voiceprint Verification Model** - classifies which member a voice sample belongs to, from `audio_features.csv`.
3. **Product Recommendation Model** - predicts `product_category` from `merged_customer_data.csv`.

All three use `RandomForestClassifier` and are evaluated with **accuracy**, **macro F1**, and **log loss**.

**Key design decisions (see prior task reviews for the reasoning):**
- Face/Voice models: `source_file`/`member`/`expression`/`phrase`/`augmentation` are metadata, not features - dropped from X, with `member` as the target.
- Voice model: `duration` is also dropped from the features. It's not a real voiceprint characteristic - just an artifact of how long that particular take happened to run - and with only 2 source recordings per member, the model could use each recording's near-unique duration as a shortcut fingerprint instead of learning actual vocal characteristics.
- Face/Voice models: evaluated with **`StratifiedGroupKFold`** grouped by `source_file`, not a plain train/test split. Each source photo/recording produces 4 near-duplicate augmented rows: splitting them randomly across train/test would leak (a model could "recognize" the exact rotated copy of a training image, inflating accuracy) rather than actually generalizing to a new, unseen photo. Grouping keeps all versions of a given photo/recording together in the same side of every fold. With only 3 photos/member and 2 recordings/member, a single split would also be too noisy to trust - k=3 (face) and k=2 (voice) folds give one clean "leave-one-photo/recording-out per class" fold each round instead of relying on a single lucky/unlucky split (verified in advance: every fold contains all 4 classes in both train and test).
- Product model: `customer_id`, `transaction_id`, `purchase_date` (raw - already captured by the engineered `purchase_month`/`purchase_dayofweek`), and `customer_rating` are dropped from X. `customer_rating` specifically is a post-purchase satisfaction score that wouldn't exist yet at the moment of making a recommendation, so keeping it as a predictor would be circular - it stays in `merged_customer_data.csv` for reference, just excluded from the model's feature set.
- All three models cap `max_depth=12`. The face model in particular has 1155 features against as few as ~16-32 training rows per fold - unconstrained trees have more than enough room to memorize idiosyncratic feature combinations rather than generalize, so depth is capped as a basic guard against overfitting.

In [2]:
import pandas as pd
import numpy as np
import joblib
import os
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import StratifiedGroupKFold, train_test_split
from sklearn.metrics import accuracy_score, f1_score, log_loss, classification_report

DATA_DIR = '../data/processed'
MODELS_DIR = '../models'
os.makedirs(MODELS_DIR, exist_ok=True)

RANDOM_STATE = 42

def run_group_cv(X, y, groups, n_splits, model_kwargs):
    """Group-aware stratified CV: fresh model per fold, returns per-fold metrics."""
    sgkf = StratifiedGroupKFold(n_splits=n_splits, shuffle=True, random_state=RANDOM_STATE)
    results = []
    for fold, (train_idx, test_idx) in enumerate(sgkf.split(X, y, groups=groups)):
        X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
        y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]

        model = RandomForestClassifier(**model_kwargs)
        model.fit(X_train, y_train)

        y_pred = model.predict(X_test)
        y_proba = model.predict_proba(X_test)

        acc = accuracy_score(y_test, y_pred)
        f1 = f1_score(y_test, y_pred, average='macro')
        loss = log_loss(y_test, y_proba, labels=model.classes_)

        results.append({'fold': fold, 'accuracy': acc, 'f1_macro': f1, 'log_loss': loss})
        print(f'fold {fold}: accuracy={acc:.3f}  f1_macro={f1:.3f}  log_loss={loss:.3f}')

    results_df = pd.DataFrame(results)
    print()
    print('mean +/- std across folds:')
    print(results_df[['accuracy', 'f1_macro', 'log_loss']].agg(['mean', 'std']))
    return results_df


## Part A: Facial Recognition Model

Target: `member`. Features: all columns except `source_file`, `member`, `expression`, `augmentation`.

In [3]:
img = pd.read_csv(os.path.join(DATA_DIR, 'image_features.csv'))

FACE_META_COLS = ['source_file', 'member', 'expression', 'augmentation']
X_face = img.drop(columns=FACE_META_COLS)
y_face = img['member']
groups_face = img['source_file']

print('X_face shape:', X_face.shape, '| classes:', sorted(y_face.unique()))

face_cv_results = run_group_cv(
    X_face, y_face, groups_face,
    n_splits=3,
    model_kwargs=dict(n_estimators=200, max_depth=12, random_state=RANDOM_STATE),
)


X_face shape: (48, 1155) | classes: ['M1', 'M2', 'M3', 'M4']
fold 0: accuracy=1.000  f1_macro=1.000  log_loss=0.367
fold 1: accuracy=0.938  f1_macro=0.937  log_loss=0.399
fold 2: accuracy=1.000  f1_macro=1.000  log_loss=0.263

mean +/- std across folds:
      accuracy  f1_macro  log_loss
mean  0.979167  0.978836  0.342649
std   0.036084  0.036657  0.071082


In [4]:
# Final deployed model: retrain on ALL available data (the CV above only estimates
# generalization performance; the demo needs a model trained on everything we have).
face_model = RandomForestClassifier(n_estimators=200, max_depth=12, random_state=RANDOM_STATE)
face_model.fit(X_face, y_face)

joblib.dump(face_model, os.path.join(MODELS_DIR, 'face_model.joblib'))
joblib.dump(list(X_face.columns), os.path.join(MODELS_DIR, 'face_feature_columns.joblib'))
print('Saved face_model.joblib, trained on all', len(X_face), 'rows.')


Saved face_model.joblib, trained on all 48 rows.


## Part B: Voiceprint Verification Model

Target: `member`. Features: all columns except `source_file`, `member`, `phrase`, `augmentation`.

In [5]:
aud = pd.read_csv(os.path.join(DATA_DIR, 'audio_features.csv'))

VOICE_META_COLS = ['source_file', 'member', 'phrase', 'augmentation']
VOICE_DROP_COLS = VOICE_META_COLS + ['duration']
X_voice = aud.drop(columns=VOICE_DROP_COLS)
y_voice = aud['member']
groups_voice = aud['source_file']

print('X_voice shape:', X_voice.shape, '| classes:', sorted(y_voice.unique()))

voice_cv_results = run_group_cv(
    X_voice, y_voice, groups_voice,
    n_splits=2,
    model_kwargs=dict(n_estimators=200, max_depth=12, random_state=RANDOM_STATE),
)


X_voice shape: (32, 34) | classes: ['M1', 'M2', 'M3', 'M4']
fold 0: accuracy=0.750  f1_macro=0.711  log_loss=0.753
fold 1: accuracy=0.938  f1_macro=0.937  log_loss=0.776

mean +/- std across folds:
      accuracy  f1_macro  log_loss
mean  0.843750   0.82381  0.764542
std   0.132583   0.15938  0.016138


In [6]:
voice_model = RandomForestClassifier(n_estimators=200, max_depth=12, random_state=RANDOM_STATE)
voice_model.fit(X_voice, y_voice)

joblib.dump(voice_model, os.path.join(MODELS_DIR, 'voice_model.joblib'))
joblib.dump(list(X_voice.columns), os.path.join(MODELS_DIR, 'voice_feature_columns.joblib'))
print('Saved voice_model.joblib, trained on all', len(X_voice), 'rows.')


Saved voice_model.joblib, trained on all 32 rows.


## Part C: Product Recommendation Model

Target: `product_category`. Features: everything except `customer_id`, `transaction_id`, `purchase_date` (raw - superseded by the engineered `purchase_month`/`purchase_dayofweek`), `customer_rating` (leakage - see note above), and the target itself. Categorical columns (`review_sentiment`, `primary_platform`) are one-hot encoded.

Unlike the face/voice models, each row here is an independent transaction (no augmented near-duplicates), so a standard stratified train/test split is appropriate - no grouping needed.

In [7]:
merged = pd.read_csv(os.path.join(DATA_DIR, 'merged_customer_data.csv'))

PROD_DROP_COLS = ['customer_id', 'transaction_id', 'purchase_date', 'customer_rating', 'product_category']
X_prod = merged.drop(columns=PROD_DROP_COLS)
X_prod = pd.get_dummies(X_prod, columns=['review_sentiment', 'primary_platform'])
y_prod = merged['product_category']

print('X_prod shape:', X_prod.shape, '| classes:', sorted(y_prod.unique()))

X_train, X_test, y_train, y_test = train_test_split(
    X_prod, y_prod, test_size=0.2, stratify=y_prod, random_state=RANDOM_STATE,
)

product_model = RandomForestClassifier(n_estimators=200, max_depth=12, random_state=RANDOM_STATE)
product_model.fit(X_train, y_train)

y_pred = product_model.predict(X_test)
y_proba = product_model.predict_proba(X_test)

acc = accuracy_score(y_test, y_pred)
f1 = f1_score(y_test, y_pred, average='macro')
loss = log_loss(y_test, y_proba, labels=product_model.classes_)

print(f'test accuracy={acc:.3f}  f1_macro={f1:.3f}  log_loss={loss:.3f}')
print()
print(classification_report(y_test, y_pred))


X_prod shape: (150, 17) | classes: ['Books', 'Clothing', 'Electronics', 'Groceries', 'Sports']
test accuracy=0.167  f1_macro=0.163  log_loss=1.955

              precision    recall  f1-score   support

       Books       0.00      0.00      0.00         5
    Clothing       0.25      0.33      0.29         6
 Electronics       0.20      0.14      0.17         7
   Groceries       0.17      0.20      0.18         5
      Sports       0.25      0.14      0.18         7

    accuracy                           0.17        30
   macro avg       0.17      0.16      0.16        30
weighted avg       0.18      0.17      0.17        30



In [8]:
# Final deployed model: retrain on all 150 rows for the demo.
product_model_final = RandomForestClassifier(n_estimators=200, max_depth=12, random_state=RANDOM_STATE)
product_model_final.fit(X_prod, y_prod)

joblib.dump(product_model_final, os.path.join(MODELS_DIR, 'product_model.joblib'))
joblib.dump(list(X_prod.columns), os.path.join(MODELS_DIR, 'product_feature_columns.joblib'))
print('Saved product_model.joblib, trained on all', len(X_prod), 'rows.')


Saved product_model.joblib, trained on all 150 rows.


## Part D: Multimodal decision logic

The three models above are evaluated individually, but the system diagram requires them to work together as a **two-factor authentication pipeline** feeding a recommendation, not three independent predictions. The combination logic (implemented in `demo/run_demo.py`) is:

1. **Face gate**: `face_model.predict_proba` must have a max confidence >= 0.70. Below that, the face isn't recognized and the flow stops (`ACCESS DENIED`) before either the voice model or the product model ever runs.
2. **Product Recommendation Model runs, but its result is held back** - it isn't shown to the user yet. This matches the system diagram's ordering (recommendation happens right after face recognition, *before* voice confirms the transaction).
3. **Voice gate**: `voice_model.predict_proba` must also have max confidence >= 0.70. Below that, `ACCESS DENIED` and the held-back product prediction is discarded, never shown.
4. **Cross-modal identity check** (beyond the assignment's minimum, added deliberately): the member identified by the face model and the member identified by the voice model must be the **same person**. Without this check, someone could pass face recognition as one member and voice verification as a *different* member and still get through - two independently-valid credentials from two different people isn't the same as one person authenticating twice. Only if both checks pass on the same identity does the held-back prediction get revealed.

**Why two factors and not one:** either model alone is a single point of failure - the face model gets fooled by nothing here since it's fairly strong (97.9% CV accuracy), but the voice model is weaker (84.4%) and, as shown below, less reliable on genuinely new samples. Requiring *both* to agree, on the *same* identity, is meaningfully harder to fool than either check alone, even with each individual model's imperfections.

The demonstration below runs this exact logic (imported from `demo/feature_extraction.py`, not reimplemented) against one legitimate case and one unauthorized case, directly in this notebook.

In [9]:
import sys
sys.path.insert(0, os.path.join('..', 'demo'))
from feature_extraction import image_path_to_features, audio_path_to_features

THRESHOLD = 0.70

def multimodal_check(image_path, audio_path):
    face_feats = image_path_to_features(image_path)
    Xf = pd.DataFrame([face_feats])[list(X_face.columns)]
    face_proba = face_model.predict_proba(Xf)[0]
    face_member = face_model.classes_[face_proba.argmax()]
    face_conf = face_proba.max()
    print(f'  Face gate:  closest match {face_member} (confidence {face_conf:.3f}, threshold {THRESHOLD})')
    if face_conf < THRESHOLD:
        print('  -> ACCESS DENIED at face stage')
        return

    audio_feats = audio_path_to_features(audio_path)
    Xv = pd.DataFrame([audio_feats])[list(X_voice.columns)]
    voice_proba = voice_model.predict_proba(Xv)[0]
    voice_member = voice_model.classes_[voice_proba.argmax()]
    voice_conf = voice_proba.max()
    print(f'  Voice gate: closest match {voice_member} (confidence {voice_conf:.3f}, threshold {THRESHOLD})')
    if voice_conf < THRESHOLD:
        print('  -> ACCESS DENIED at voice stage')
        return
    if voice_member != face_member:
        print(f'  -> ACCESS DENIED: face ({face_member}) and voice ({voice_member}) identities do not match')
        return

    print(f'  -> ACCESS GRANTED, identity confirmed as {face_member}')

print('=== Case 1: legitimate M3 (own face + own voice) ===')
multimodal_check('../image_data/M3_1.jpeg', '../audio_data/M3_approve.ogg')

print()
print('=== Case 2: unauthorized face ===')
multimodal_check('../unauthorized_data/unauthorized_face.jpeg', '../audio_data/M1_approve.mp4')


=== Case 1: legitimate M3 (own face + own voice) ===
  Face gate:  closest match M3 (confidence 0.990, threshold 0.7)
  Voice gate: closest match M3 (confidence 0.965, threshold 0.7)
  -> ACCESS GRANTED, identity confirmed as M3

=== Case 2: unauthorized face ===
  Face gate:  closest match M4 (confidence 0.660, threshold 0.7)
  -> ACCESS DENIED at face stage


## Summary

Saved to `../models/`:
- `face_model.joblib` + `face_feature_columns.joblib`
- `voice_model.joblib` + `voice_feature_columns.joblib`
- `product_model.joblib` + `product_feature_columns.joblib`

The `*_feature_columns.joblib` files record the exact column order/set each model was trained on, so the Task 5 demo can build a matching feature vector for new, unseen input (e.g. one-hot columns for a `review_sentiment` value that doesn't appear in a given inference row).